# Module 1: From "It Went Down" to "The Program Did It"

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

The Beginner series argued that a decline after a program is not the same as a
decline caused by it. This module turns that into arithmetic.

Three sentences about the same program, each stronger than the last, each
requiring one more piece of evidence. By the end you will have computed all
three and be able to say which one a given set of records supports.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

## 2. Rung one: the rate fell

The weakest sentence, and the only one that needs nothing but the agencies'
own records.

In [ ]:
before = cell_rate(TRAINED, "before")
after = cell_rate(TRAINED, "after")
print(f"  trained agencies, before the program: {before:.2f} per 100 arrests")
print(f"  trained agencies, after it was in place: {after:.2f}")
print(f"  change: {100 * (after / before - 1):+.1f} percent")

Thirty three percent. This sentence is true and it is about the agencies, not
about the program.

## 3. Rung two: it fell more than at other agencies

Now bring in the seven agencies that never took the training. The comparison
tells you how much of the fall was going to happen anyway.

In [ ]:
c_before = cell_rate(COMPARISON, "before")
c_after = cell_rate(COMPARISON, "after")

tab = pd.DataFrame(
    {"before": [before, c_before], "after": [after, c_after]},
    index=["took the training", "did not"])
tab["change"] = (100 * (tab["after"] / tab["before"] - 1)).round(1).astype(str) + "%"
print(tab.round(2).to_string())

did = 100 * ((after / before) / (c_after / c_before) - 1)
print(f"\n  one change divided by the other: {did:+.1f} percent")

The comparison agencies fell 19.5 percent without doing anything. Dividing one
change by the other leaves **16.9 percent** for the program.

Half the apparent effect has just disappeared, and nothing was wrong with the
first calculation. It answered a different question.

## 4. Rung three: it fell more than at agencies that were moving the same way

Rung two assumes the two groups would have changed by the same amount. That is
checkable, in the years before the program existed.

In [ ]:
pre = f[f["period"] == "before"].copy()
idx = pd.PeriodIndex(pre["year_month"], freq="M")
pre["yr"] = idx.year.values + (idx.month.values - 1) / 12.0

import statsmodels.api as sm
import statsmodels.formula.api as smf

print("  annual change in the use of force rate, before the program existed:")
for label, sub in [("the seven comparison agencies", pre[pre["trained"] == 0]),
                   *[(f"  {NAME[a].split()[0]}", pre[pre["agency_id"] == a])
                     for a in TRAINED]]:
    z = smf.glm("n_uof ~ yr", sub, family=sm.families.Poisson(),
                offset=np.log(sub["n_arrests"])).fit()
    print(f"  {label:32s} {100 * (np.exp(z.params['yr']) - 1):+6.2f} percent a year")

Four of the five trained agencies sit close to the comparison group's 4.5
percent a year. **Summit County was falling at 12 percent a year before the
training existed**, for reasons of its own.

Rung three uses only the agencies for which the assumption survives.

In [ ]:
keep = [a for a in TRAINED if a != "A007"]
b4, a4 = cell_rate(keep, "before"), cell_rate(keep, "after")
did3 = 100 * ((a4 / b4) / (c_after / c_before) - 1)

pd.DataFrame([
    {"sentence": "the rate fell",
     "estimate": f"{100 * (after / before - 1):+.1f}%",
     "what it needs": "records"},
    {"sentence": "it fell more than at other agencies",
     "estimate": f"{did:+.1f}%",
     "what it needs": "records from seven more agencies"},
    {"sentence": "it fell more than at agencies moving the same way",
     "estimate": f"{did3:+.1f}%",
     "what it needs": "the same, plus a pre period check"},
    {"sentence": "THE TRUTH", "estimate": f"{TRUTH:+.1f}%", "what it needs": ""},
]).set_index("sentence")

Each rung costs one more piece of evidence and moves the number closer. The
third one lands within half a point of the truth.

**Notice what did not happen.** Nothing became more sophisticated. The third
estimate is the same division as the second, run on a subset chosen by a check
that anyone can do with a plot.

## 5. The rung above, and why this module stops here

There is a fourth sentence: *the training caused a 12 percent reduction*. Rung
three does not reach it, because the division removes only what the two groups
had **in common**. Anything that happened to the trained agencies alone is
still inside the number.

| Still unresolved after rung three | Where it is taken up |
|---|---|
| Why these five agencies were chosen | Module 10 |
| Whether a comparison agency was quietly trained | Module 13 |
| Whether the two groups would really have moved alike | Modules 7 and 8 |
| How much of this could be luck | Module 14 |

And a fifth sentence, *the training would work elsewhere*, which no single
study reaches.

## Exercise

Rung two used all seven comparison agencies pooled. Try each one alone and see
how much the answer depends on the choice.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for a in COMPARISON:
        cb, ca = cell_rate([a], "before"), cell_rate([a], "after")
        rows.append({"comparison agency": NAME[a],
                     "its own change": f"{100 * (ca / cb - 1):+.1f}%",
                     "estimate it produces":
                         f"{100 * ((a4 / b4) / (ca / cb) - 1):+.1f}%"})
    rows.append({"comparison agency": "ALL SEVEN POOLED",
                 "its own change": f"{100 * (c_after / c_before - 1):+.1f}%",
                 "estimate it produces": f"{did3:+.1f}%"})
    print(f"  the truth is {TRUTH:+.1f} percent\n")
    display(pd.DataFrame(rows).set_index("comparison agency"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

The seven single agency estimates run from **minus 29.3 percent to plus 2.3
percent**. One of them has the wrong sign: compared against Prairie County
alone, the training appears to have made things slightly **worse**. The pooled
estimate is 12.5, close to the truth, and no single agency is reliably near
it.

The reason is visible in the middle column. Prairie County's own rate fell
31.1 percent over the same period, more than any trained agency, so dividing
by it wipes the effect out. Dunmoor's barely moved at all, so dividing by it
inflates the effect to 29 percent. Neither agency did anything unusual; both
are small enough that their three year change is mostly accident.

**This is the argument for pooling rather than picking.** A single comparison
agency carries its own accidents into the answer. Averaging seven agencies
averages seven sets of accidents.

It is also the argument for fixing the rule before looking. Any of these seven
answers could be reported honestly, and an analyst who tried all seven and
reported the most convincing one has done something quite different from an
analyst who declared the rule in advance.

</details>

---

**Next:** [Module 2: Potential Outcomes Without the Algebra](Module_02_Potential_Outcomes_Without_The_Algebra.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*